**Data Cleaning and Processing**

EPIC - RL2 - SP3

Paulo Yoshio Kuga

In [1]:
import lissa as li
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import json

The one-hour frequency file is supposed to be loaded into base_data. Then, data has its types infered and time is defined as time. 

Nonetheless, the duplicates are delete and the data is exported.

But it's expected to work in any file that has the same headers that what came for us in the aforementioned file.

In [2]:
file_dir = "../data"
file_name = "full_sensor_1h [old].csv"

# Path to file
file_path = file_dir + "/" + file_name

# Read data and convert to respective type
base_data = pd.read_csv(file_path, delimiter=",")
base_data = base_data.infer_objects(copy=False)
base_data["time"] = pd.to_datetime(base_data["time"])
base_data["Well"] = base_data["Well Run"].str[:4]

print(
    f"Pumps: #{base_data['Well'].unique().size}\n"
    f"Runs: #{base_data['Well Run'].unique().size}"
)


# Identify and remove duplicate rows
columns_for_dup_check = list(set(list(base_data)) - {"Well_down", "Well Run"})
dup = base_data[columns_for_dup_check].duplicated()
base_data.drop(index=base_data[dup].index, columns="Well", inplace=True)

#base_data.set_index("time").to_csv("../data/base_data.csv")

Pumps: #38
Runs: #57


Then, the file respostas.csv is called. respostas.csv is an PEREGRINO ESP Run Life fail spreadsheet.

This file is loaded with the intent of allowing our data be linked with faliure date.

A column, called "Well_down" was created by Equinor, analyzing if the well was on off or online. 

Having the faliure date at main data, it is possible to realize a comparision between the dates and infer if there will be a faliure or not. Therefore, with this approach, we can label all data, if it's relatable with a faliure or not.

However, this approach does not point exactly when it was observed the faliure, but, in reality, only labels if the proababilty of the anomaly is inside the subset is greater.

In [3]:
failure_file = "respostas.csv"

filePath = file_dir + "/" + failure_file #path to file
respostas = pd.read_csv(filePath, delimiter=",") #import faliure data

# Filter all pump names in the dataset
pump_list = base_data["Well Run"].unique()

# Only analyze data that is in the time dataset
respostas = respostas.loc[respostas["Well Run"].isin(pump_list)]

# Failure precise time is not considered in the dataset, since it's only a report
respostas["Failure Date"] = pd.to_datetime(
    respostas["Failure Date"],
    format="%d/%m/%y"
).dt.tz_localize('UTC')

# Do a left merge to unite failure date with time info
entire_data = pd.merge(base_data, respostas, how="left", on="Well Run")

# Check if failure already happened
entire_data["Failure"] = entire_data["time"] >= entire_data["Failure Date"]

# Remove duplicated columns
drop_list = ['Start-up Date', 'Start-up Date', 'ESP Failure?', 'Failure Date']
entire_data.drop(drop_list, axis=1, inplace=True)

# Time is set as index
entire_data.set_index("time", inplace=True)

 This function is intended to process columns for feature creation. 

Currently is only synthetizing current A,B,C in a norm-2, doing the same for vibrations and converting columns written as 1/100 units in fractions.

Users might want to deal with new features. It is not the best pratice, but they should be written here (inside the function).

In [4]:
entire_data["ESP Current Module"] = (
entire_data["ESP motor Current - phase A"].pow(2)+
entire_data["ESP motor Current - phase B"].pow(2)+
entire_data["ESP motor Current - phase C"].pow(2)).pow(1/2)

entire_data["ESP Vibration Module"] = (entire_data["ESP Vibration X"].pow(2)+entire_data["ESP Vibration Y"].pow(2)).pow(1/2)

entire_data["ESP Motor Voltage"] = entire_data["ESP Motor Voltage"]/1000

entire_data["Choke Opening"] = entire_data["Choke Opening"]/100
entire_data["Water Cut @ 20degC - 1 atm"] = entire_data["Water Cut @ 20degC - 1 atm"]/100


entire_data.drop(columns=[
    # "Current Mean",
    "ESP motor Current - phase A",
    "ESP motor Current - phase B",
    "ESP motor Current - phase C",
    "ESP Vibration X",
    "ESP Vibration Y",
    'ESP differential pressure', #we are going to try to find this in PCA
    "ESP discharge temperature sensor" #there's too many few entries to be considered.

],inplace=True)

#entire_data.to_csv("../data/entire_data.csv")

In [5]:
with open("./dictionaries/headers.json","r") as dictionary:
    headers = json.load(dictionary)

numerical_headers = list(set(entire_data.columns)-set(headers["information"])-set(headers["percentual"]))
numerical_headers.sort()
new_dict = {
    "numerical_headers":numerical_headers,
    "non_numerical_headers": headers["information"]
    }

with open("./dictionaries/new_headers.json", "w") as file:
    json.dump(new_dict, file, indent=4)

For each pump, for selected numerical properties, a low frequency filter is passed, trying to reduce the noise into the data. 

In [6]:
total_data = pd.DataFrame(columns=list(entire_data))

for pump in pump_list:
    # Copies the original dataset
    export_data = entire_data.loc[entire_data["Well Run"] == pump].copy()
    
    # Apply exponential weighted mean and standardization filter
    filter_data = export_data.groupby("Well_down")[numerical_headers].apply(
        lambda x: (x.ewm(span=24).mean() - x.expanding().median()) / x.expanding().std()
    )
    
    filter_data = (
        filter_data
        .reset_index()
        .set_index("time")
        .fillna(0)
        .sort_index()
        .drop(columns="Well_down")
    )
    
    total_data = pd.concat(
        [
            total_data,
            pd.merge(filter_data, export_data[headers["information"]+headers["percentual"]], how="left", on="time")
        ],
        axis=0
    )

#total_data.reset_index(inplace=True)
#total_data.to_csv("../data/totalProcessedData.csv")
#total_data.set_index("index", inplace=True)

/tmp/ipykernel_284192/241539414.py:21: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  total_data = pd.concat(
/tmp/ipykernel_284192/241539414.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  total_data = pd.concat(
